# Projeto 1: Regressão com MLP

O código de dados, modelos, métricas e figuras está em `src/`. Este notebook reproduz a ablação (baseline, dropout, L2, L1 e momentum).


In [ ]:
from src.config import (
    BATCH_SIZE,
    DROPOUT,
    L1_LAMBDA,
    L2_WEIGHT_DECAY,
    LEARNING_RATE,
    MOMENTUM,
    NEURONS,
    NUM_EPOCHS,
    NUM_LAYERS,
)
from src.data import prepare_splits
from src.evaluate import comparison_table
from src.model import MLP, MLP_DROPOUT, MLP_L1, MLP_L2, MLP_MOMENTUM
from src.plots import plot_loss, plot_train_predictions, save_report_figures


In [ ]:
splits = prepare_splits()
X_train, y_train = splits.X_train, splits.y_train
X_val, y_val = splits.X_val, splits.y_val
X_test, y_test = splits.X_test, splits.y_test


# Definição do modelo baseline


In [ ]:
model = MLP(num_layers=NUM_LAYERS, neurons=NEURONS)
model


In [ ]:
train_losses, val_losses = model.fit(
    X_train, y_train, X_val, y_val,
    num_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE,
)

plot_loss(train_losses, val_losses)
print(train_losses[-1], val_losses[-1])
print("gap:", val_losses[-1] - train_losses[-1])


In [ ]:
train_preds = model.predict(X_train)
plot_train_predictions(X_train, y_train, train_preds)


# Definição do modelo com Dropout


In [ ]:
model_dropout = MLP_DROPOUT(num_layers=NUM_LAYERS, neurons=NEURONS, dropout=DROPOUT)
model_dropout


In [ ]:
dropout_train_losses, dropout_val_losses = model_dropout.fit(
    X_train, y_train, X_val, y_val,
    num_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE,
)

plot_loss(dropout_train_losses, dropout_val_losses, xlabel="Epoch", ylabel="MSE")
print(dropout_train_losses[-1], dropout_val_losses[-1])
print("gap:", dropout_val_losses[-1] - dropout_train_losses[-1])


In [ ]:
dropout_train_preds = model_dropout.predict(X_train)
plot_train_predictions(X_train, y_train, dropout_train_preds)


# Regularização L2


In [ ]:
model_l2 = MLP_L2(num_layers=NUM_LAYERS, neurons=NEURONS)
model_l2


In [ ]:
l2_train_losses, l2_val_losses = model_l2.fit(
    X_train, y_train, X_val, y_val,
    num_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE,
    weight_decay=L2_WEIGHT_DECAY,
)

plot_loss(l2_train_losses, l2_val_losses, xlabel="Epoch", ylabel="MSE")
print(l2_train_losses[-1], l2_val_losses[-1])
print("gap:", l2_val_losses[-1] - l2_train_losses[-1])


In [ ]:
l2_train_preds = model_l2.predict(X_train)
plot_train_predictions(X_train, y_train, l2_train_preds)


# Modelo com regularização L1


In [ ]:
model_l1 = MLP_L1(num_layers=NUM_LAYERS, neurons=NEURONS)
model_l1


In [ ]:
l1_train_losses, l1_val_losses = model_l1.fit(
    X_train, y_train, X_val, y_val,
    num_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE,
    l1=L1_LAMBDA,
)

plot_loss(l1_train_losses, l1_val_losses, xlabel="Epoch", ylabel="MSE")
print(l1_train_losses[-1], l1_val_losses[-1])
print("gap:", l1_val_losses[-1] - l1_train_losses[-1])


In [ ]:
l1_train_preds = model_l1.predict(X_train)
plot_train_predictions(X_train, y_train, l1_train_preds)


# Modelo com Momentum


In [ ]:
model_mom = MLP_MOMENTUM(num_layers=NUM_LAYERS, neurons=NEURONS)
model_mom


In [ ]:
mom_train_losses, mom_val_losses = model_mom.fit(
    X_train, y_train, X_val, y_val,
    num_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE,
    momentum=MOMENTUM,
)

plot_loss(mom_train_losses, mom_val_losses, xlabel="Epoch", ylabel="MSE")
print(mom_train_losses[-1], mom_val_losses[-1])
print("gap:", mom_val_losses[-1] - mom_train_losses[-1])


# Métricas no teste


In [ ]:
experiments = [
    ("Baseline", model, val_losses),
    ("Dropout", model_dropout, dropout_val_losses),
    ("L2", model_l2, l2_val_losses),
    ("L1", model_l1, l1_val_losses),
    ("Momentum", model_mom, mom_val_losses),
]

df_test = comparison_table(experiments, splits)
df_test.round(4)


# Figuras para o relatório

Curvas de perda, *parity plot* e resíduos no teste. Os PNGs são gravados em `report/redes_neurais_template_projetos/figures/`.


In [ ]:
results = [
    {"name": "Baseline", "label": "Baseline (SGD vanilla)", "model": model, "train_losses": train_losses, "val_losses": val_losses},
    {"name": "Dropout", "label": r"Dropout ($p=0.05$)", "model": model_dropout, "train_losses": dropout_train_losses, "val_losses": dropout_val_losses},
    {"name": "L2", "label": r"L2 ($\lambda=10^{-3}$)", "model": model_l2, "train_losses": l2_train_losses, "val_losses": l2_val_losses},
    {"name": "L1", "label": r"L1 ($\lambda=10^{-4}$)", "model": model_l1, "train_losses": l1_train_losses, "val_losses": l1_val_losses},
    {"name": "Momentum", "label": r"Momentum ($\beta=0.9$)", "model": model_mom, "train_losses": mom_train_losses, "val_losses": mom_val_losses},
]

fig_dir = save_report_figures(splits, results, show=True)
print("Figuras salvas em", fig_dir.resolve())
